# Analyse intégrale du firmware IHU007 / IHU007B (lecture seule)

**Objectif :** parcourir le contenu complet de `boot.img`, `system.img` et `vendor.img` dans l'archive `IHU007 non-modified.rar` déjà stockée dans **votre propre Google Drive**. Aucun flash, aucune installation automobile, aucun changement dans le firmware.

### Utilisation sur téléphone ou ordinateur

1. Ouvrir ce notebook dans [Google Colab](https://colab.research.google.com/) (menu **Fichier → Importer un notebook**, choisir le fichier `.ipynb`). Sur téléphone, activer éventuellement **Version pour ordinateur** dans Chrome.
2. Sélectionner **Exécution → Tout exécuter**, puis autoriser l'accès à **votre propre Drive** dans l'interface officielle de Colab.
3. Attendre la fin de l'analyse : les rapports sont écrits dans **Mon Drive / IHU007_ANALYSE_RAPPORTS**. Revenir dans ChatGPT et dire « analyse terminée » ; le dossier peut être lu via la connexion Drive.

Le traitement peut demander plusieurs dizaines de minutes et beaucoup de disque temporaire (au moins 25 Go libres). Il s'effectue dans votre session Colab, sans partager publiquement l'archive. **Ne pas déplacer/supprimer les images d'origine.** Si un format d'image n'est pas pris en charge, le rapport le signalera explicitement. Les fichiers de numéros de série, clés et secrets ne sont pas copiés dans les paquets de restitution.

La phase automatique recense et vérifie **chaque fichier extrait** (chemin, taille, empreinte SHA-256, signature, correspondances techniques). Pour les APK/JAR, elle inventorie également les fichiers internes et relève les métadonnées accessibles ; ce n'est pas une décompilation exhaustive de tous les binaires. L'analyse humaine détaillée se fera ensuite à partir des rapports et composants prioritaires.


In [ ]:
# @title Lancer l'analyse complète de IHU007 non-modified.rar
from google.colab import drive
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv, gzip, hashlib, io, json, os, re, shutil, struct, subprocess, sys, time, traceback, zipfile

def run(cmd, *, check=True, cwd=None, quiet=False, timeout=None):
    print('$', ' '.join(map(str,cmd)), flush=True)
    p=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    if not quiet and p.stdout: print(p.stdout[-3500:],flush=True)
    if check and p.returncode: raise RuntimeError('Échec '+str(p.returncode)+': '+' '.join(map(str,cmd))+'\n'+p.stdout[-5000:])
    return p

def install_tools():
    run(['apt-get','update','-qq'],check=False,quiet=True)
    for pkg in ['unar','p7zip-full','e2fsprogs','android-sdk-libsparse-utils','file','binutils','libarchive-tools','squashfs-tools','erofs-utils','cpio','aapt']:
        p=run(['apt-get','install','-y','-qq',pkg],check=False,quiet=True)
        print('Outil',pkg,':','installé' if p.returncode==0 else 'non disponible, solution alternative si possible')

install_tools()
drive.mount('/content/drive')
mydrive=Path('/content/drive/MyDrive')
archive=mydrive/'IHU007 non-modified.rar'
if not archive.exists():
    candidates=[p for p in mydrive.rglob('*') if p.is_file() and p.name.lower()=='ihu007 non-modified.rar']
    if len(candidates)!=1: raise FileNotFoundError('Archive introuvable ou ambiguë dans Mon Drive : '+str(candidates[:5]))
    archive=candidates[0]
print('Archive trouvée :',archive,'taille=',archive.stat().st_size,'octets')
if archive.stat().st_size<1_000_000_000: print('ATTENTION : archive plus petite que celle annoncée, vérifier intégrité.')

free=shutil.disk_usage('/content').free
print('Espace disque temporaire libre :',round(free/2**30,1),'Gio')
if free<22*2**30: raise RuntimeError('Espace Colab insuffisant (<22 Gio). Choisir un environnement disposant de >=25 Gio libres.')
work=Path('/content/ihu007_complet'); raw=work/'images'; unpacked=work/'partitions'; reports=work/'rapports'; assets=work/'composants'
for p in [raw,unpacked,reports,assets]: p.mkdir(parents=True,exist_ok=True)
errors=[]; notes=[]; report_id=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')

def safe_write(name,txt): (reports/name).write_text(str(txt),encoding='utf-8')
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
    return h.hexdigest()

# D'abord extraction de TOUS les membres de l'archive RAR.
print('\n===== EXTRACTION RAR INTÉGRALE =====')
p=run(['unar','-force-overwrite','-quiet','-output-directory',str(raw),str(archive)],check=False,quiet=True)
if p.returncode:
    print('unar a échoué, deuxième tentative avec 7z...')
    p2=run(['7z','x','-y','-o'+str(raw),str(archive)],check=False,quiet=True)
    if p2.returncode: raise RuntimeError('Extraction RAR impossible. unar: '+p.stdout[-2500:]+'; 7z: '+p2.stdout[-2500:])
all_images={p.name.lower():p for p in raw.rglob('*') if p.is_file() and p.name.lower().endswith('.img')}
others=[p for p in raw.rglob('*') if p.is_file() and not p.name.lower().endswith('.img')]
if not all_images: raise RuntimeError('Aucune image .img dans l’archive — vérifier format et mot de passe.')
print('Images :',[(k,round(v.stat().st_size/2**20,1)) for k,v in all_images.items()]);print('Autres membres:',[p.name for p in others])
with (reports/'membres_archive.csv').open('w',newline='',encoding='utf-8') as fo:
    w=csv.writer(fo);w.writerow(['chemin_dans_archive','taille_octets','sha256'])
    for p in sorted(x for x in raw.rglob('*') if x.is_file()):
        # SN.jpg peut comporter un identifiant : on note son existence et son empreinte mais ne l’exporte pas.
        w.writerow([str(p.relative_to(raw)),p.stat().st_size,sha256(p)])

# Conversion seulement si l'image est Android Sparse.
def is_sparse(path):
    with path.open('rb') as f:return f.read(4)==b'\x3a\xff\x26\xed'

def is_ext4(path):
    with path.open('rb') as f:f.seek(1080);return f.read(2)==b'\x53\xef'

def kind(path):
    with path.open('rb') as f:
        head=f.read(2048)
    if head.startswith(b'ANDROID!'):return 'boot-android'
    if head.startswith(b'hsqs'):return 'squashfs'
    if head[1024:1028]==b'\xe2\xe1\xf5\xe0':return 'erofs'
    if head[1080:1082]==b'\x53\xef':return 'ext4'
    return 'inconnu: '+run(['file','-b',str(path)],check=False,quiet=True).stdout.strip()[:180]

# Extraction contrôlée boot.img/ramdisk sans exécuter de code issu de l'image.
def extract_boot(path):
    dest=unpacked/'boot_ramdisk'; dest.mkdir(exist_ok=True)
    with path.open('rb') as f:
        header=f.read(4096)
        if not header.startswith(b'ANDROID!'):
            errors.append('boot.img : en-tête Android absent');return
        ks,ka,rs,ra,ss,sa,tags,page,ver,osver=struct.unpack_from('<10I',header,8)
        cmdline=header[64:64+512].split(b'\x00')[0].decode('utf-8','replace')
        ext=header[1632:1632+1024].split(b'\x00')[0].decode('utf-8','replace') if len(header)>1632 else ''
        summary={'kernel_size':ks,'ramdisk_size':rs,'page_size':page,'header_version':ver,'os_version_encoded':osver,'cmdline':cmdline,'extra_cmdline_apercu':ext[:500]}
        safe_write('boot_en_tete.json',json.dumps(summary,ensure_ascii=False,indent=2))
        if page<=0 or page>65536 or rs>200*2**20:errors.append('boot.img : tailles hors limites');return
        f.seek(page)
        kernel=f.read(ks)
        kernel_folder=unpacked/'boot_kernel'
        kernel_folder.mkdir(exist_ok=True)
        (kernel_folder/'kernel.bin').write_bytes(kernel)
        f.seek(page*(1+(ks+page-1)//page));ram=f.read(rs)
    if ram.startswith(b'\x1f\x8b'):
        content=gzip.decompress(ram)
    else:
        errors.append('Ramdisk : compression non prise en charge, '+ram[:12].hex());return
    p=run(['cpio','--extract','--make-directories','--no-absolute-filenames','--unconditional'],check=False,quiet=True,cwd=str(dest)) if False else None
    # subprocess.stdin binaire pour CPIO ; le chemin sortant est confiné au répertoire temporaire.
    proc=subprocess.run(['cpio','-idm','--no-absolute-filenames'],input=content,cwd=str(dest),stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    if proc.returncode: errors.append('Extraction CPIO boot : '+proc.stderr.decode('utf-8','replace')[-700:])

print('\n===== ANALYSE DES IMAGES (LECTURE SEULE) =====')
image_stats=[]
for name, original in sorted(all_images.items()):
    row={'name':name,'original_bytes':original.stat().st_size,'sha256':sha256(original),'sparse':is_sparse(original)}
    usable=original
    if is_sparse(original):
        usable=work/('raw_'+name)
        try:run(['simg2img',str(original),str(usable)],quiet=True)
        except Exception as e:errors.append(f'{name}: conversion sparse impossible: {e}');image_stats.append(row);continue
    k=kind(usable);row.update({'filesystem':k,'usable_bytes':usable.stat().st_size});image_stats.append(row)
    try:
        if k=='boot-android':extract_boot(usable)
        elif k=='ext4':
            target=unpacked/name.replace('.img','');target.mkdir(parents=True,exist_ok=True)
            # debugfs lit directement ext4 sans droits de montage et sans modifier l'image.
            p=run(['debugfs','-R','rdump / '+str(target),str(usable)],check=False,quiet=True)
            if p.returncode: errors.append(name+': debugfs a échoué: '+p.stdout[-1400:])
        elif k=='erofs':
            target=unpacked/name.replace('.img','');target.mkdir(parents=True,exist_ok=True)
            p=run(['fsck.erofs','--extract='+str(target),str(usable)],check=False,quiet=True)
            if p.returncode: errors.append(name+': EROFS non extrait: '+p.stdout[-1000:])
        elif k=='squashfs':
            target=unpacked/name.replace('.img','')
            p=run(['unsquashfs','-f','-d',str(target),str(usable)],check=False,quiet=True)
            if p.returncode:errors.append(name+': squashfs non extrait: '+p.stdout[-1000:])
        else:errors.append(name+': format non reconnu, pas d’extraction ; voir signature.')
    except Exception as e:errors.append(name+': '+str(e)[:500])
safe_write('images_manifest.json',json.dumps(image_stats,indent=2,ensure_ascii=False))

# INVENTAIRE INTÉGRAL : tous les fichiers extraits, y compris sans mot-clé particulier.
KEYWORDS=[b'ecarx',b'adaptapi',b'ihu007',b'carfunction',b'carsensor',b'isensor',b'getsensorlatestvalue',b'getendurancemileage',b'bcmmodule',b'climate',b'vr_res',b'gocsdk',b'audiorecord',b'voice',b'bluetooth',b'vehicle',b'canbus',b'android.car',b'mark99',b'lunaris']
def magic(buf):
    if buf.startswith(b'\x7fELF'):return 'ELF'
    if buf.startswith(b'PK\x03\x04'):return 'ZIP/APK/JAR'
    if buf.startswith(b'dex\n'):return 'DEX'
    if buf.startswith(b'\x89PNG'):return 'PNG'
    if buf.startswith(b'\xff\xd8\xff'):return 'JPEG'
    if buf.startswith(b'SQLite format 3'):return 'SQLite'
    if buf.startswith(b'\xca\xfe\xba\xbe'):return 'Java class'
    return 'autre'

def hash_and_hits(path):
    h=hashlib.sha256();hits=Counter();previous=b''; first=None
    with path.open('rb') as f:
        while True:
            part=f.read(1024*1024)
            if not part:break
            if first is None:first=part[:32]
            h.update(part)
            low=(previous+part).lower()
            for kw in KEYWORDS:
                c=low.count(kw)
                if c:hits[kw.decode()]+=c
            previous=part[-63:]
    return h.hexdigest(),magic(first or b''),hits

print('\n===== INDEXATION EXHAUSTIVE DES FICHIERS =====')
records=[]; hits_rows=[]; target_paths=[]; candidates=[]
sensitive_re=re.compile(r'(^|/)(sn\.|imei|serial|account|password|passwd|keystore|private.?key|credentials?|userdata|\.ssh|id_rsa|\.git)(/|\.|$)',re.I)
relevant_re=re.compile(r'ecarx|adaptapi|carservice|carcontrol|vehicle|voice|vr_res|bluetooth|gocsdk|gkui|media|sensor|climate|phone|dialer|vrengine|speech',re.I)
for root,ds,fs in os.walk(unpacked,followlinks=False):
    for fn in sorted(fs):
        p=Path(root)/fn; rel=p.relative_to(unpacked).as_posix()
        try:
            if p.is_symlink():
                records.append([rel,0,'symlink', '', '']);continue
            if not p.is_file():continue
            hs,typ,hit=hash_and_hits(p)
            records.append([rel,p.stat().st_size,typ,hs,';'.join(f'{k}:{v}' for k,v in hit.items())])
            if hit:hits_rows.append([rel,p.stat().st_size,';'.join(f'{k}:{v}' for k,v in hit.items())])
            if relevant_re.search(rel) or any(h in hit for h in ['ecarx','adaptapi','carsensor','getsensorlatestvalue','ihu007']):candidates.append((p,rel))
        except Exception as e:errors.append('Inventaire '+rel+': '+str(e)[:150])
with (reports/'tous_les_fichiers.csv').open('w',newline='',encoding='utf-8') as fo:
    w=csv.writer(fo);w.writerow(['chemin','taille_octets','type','sha256','mentions']);w.writerows(records)
with (reports/'references_automobiles_et_voix.csv').open('w',newline='',encoding='utf-8') as fo:
    w=csv.writer(fo);w.writerow(['chemin','taille_octets','mentions']);w.writerows(hits_rows)
print('Fichiers inventoriés :',len(records),'dont',len(hits_rows),'avec indices techniques')

# INVENTAIRE TOUS LES CONTENUS APK/JAR/APEX/ZIP : liste, manifeste Android, DEX/ELF internes.
print('\n===== INDEXATION DES ARCHIVES ANDROID EMBARQUÉES =====')
app_rows=[]; archive_rows=[]; archive_hits=[]
aapt=shutil.which('aapt')
for row in records:
    rel,size,typ,hs,hit=row
    p=unpacked/rel
    if not p.is_file() or not rel.lower().endswith(('.apk','.jar','.apex','.zip')):continue
    info=''
    if rel.lower().endswith('.apk') and aapt:
        try:
            x=subprocess.run([aapt,'dump','badging',str(p)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=35)
            match=re.search(r"package: name='([^']+)' versionCode='([^']*)' versionName='([^']*)'",x.stdout)
            info=' | '.join(match.groups()) if match else 'manifeste_illisible'
        except Exception as e:
            info='erreur_aapt';errors.append('AAPT '+rel+': '+str(e)[:150])
    app_rows.append([rel,size,info])
    try:
        with zipfile.ZipFile(p) as ar:
            for infoobj in ar.infolist():
                archive_rows.append([rel,infoobj.filename,infoobj.file_size,infoobj.compress_size])
                # Analyse statique interne ciblée pour DEX et XML, sans exécuter les APK.
                if (infoobj.filename.endswith(('.dex','.xml','.so','.class')) and infoobj.file_size<120*2**20):
                    counts=Counter()
                    try:
                        with ar.open(infoobj) as f:
                            previous=b''
                            for part in iter(lambda:f.read(1024*1024),b''):
                                low=(previous+part).lower()
                                for kw in KEYWORDS:
                                    n=low.count(kw)
                                    if n:counts[kw.decode()]+=n
                                previous=part[-63:]
                        if counts:archive_hits.append([rel,infoobj.filename,';'.join(f'{k}:{v}' for k,v in counts.items())])
                    except Exception as e:errors.append('Entrée '+rel+'/'+infoobj.filename+': '+str(e)[:120])
    except Exception as e:errors.append('Archive interne '+rel+': '+str(e)[:150])
for name,heads,rows in [
    ('applications_et_archives.csv',['chemin','taille_octets','package_version'],app_rows),
    ('contenu_integral_apk_jar.csv',['archive','entree','taille','taille_compressee'],archive_rows),
    ('references_internes_apk_jar.csv',['archive','entree','mentions'],archive_hits)]:
    with (reports/name).open('w',newline='',encoding='utf-8') as fo:
        w=csv.writer(fo);w.writerow(heads);w.writerows(rows)

# COLLECTE NON INTRUSIVE : textes de configuration accessibles ; ne pas inclure les secrets.
props=[]; initrc=[]
for row in records:
    rel=row[0];p=unpacked/rel
    if p.is_file() and not sensitive_re.search(rel) and p.stat().st_size<8*2**20:
        if rel.endswith(('.prop','.rc')) or ('/etc/permissions/' in rel and rel.endswith('.xml')):
            try:
                content=p.read_text(encoding='utf-8',errors='replace')
                if rel.endswith('.prop'):
                    props.extend([f'{rel}\t{line}' for line in content.splitlines() if line.strip() and not re.search(r'serial|imei|secret|password|token|key=',line,re.I)])
                elif rel.endswith('.rc') and re.search('ecarx|gocsdk|vr_res|voice|bluetooth|vehicle',content,re.I):
                    initrc.extend([f'{rel}\t{line}' for line in content.splitlines() if re.search('ecarx|gocsdk|vr_res|voice|bluetooth|vehicle',line,re.I)])
            except Exception:pass
safe_write('proprietes_android.txt','\n'.join(props))
safe_write('services_rc_reperes.txt','\n'.join(initrc))

# Extrait des composants de base utiles à l'étude humaine, sans données sensibles.
# Limite chaque paquet sous 200 Mio pour récupération via la connexion ChatGPT.
print('\n===== PRÉPARATION DES PAQUETS TECHNIQUES (<200 Mio chacun) =====')
priority=[]
for p,rel in candidates:
    try:
        if sensitive_re.search(rel) or p.is_symlink() or not p.is_file():continue
        sz=p.stat().st_size
        if sz>65*2**20:continue
        score=3 if re.search('adapterapi|ecarx|gkui|carservice|carcontrol|vehicleservice|vr_res|gocsdk',rel,re.I) else 1
        priority.append((score,sz,p,rel))
    except OSError:pass
priority.sort(key=lambda x:(-x[0],x[1]))
parts=[];curr=[];total=0;cap=190*2**20
for score,sz,p,rel in priority:
    if total+sz>cap and curr:
        parts.append(curr);curr=[];total=0
    curr.append((p,rel));total+=sz
    if len(parts)>=5:break
if curr and len(parts)<5:parts.append(curr)
for i,items in enumerate(parts,1):
    zip_path=reports/f'composants_techniques_{i:02d}.zip'
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as f:
        for p,rel in items:
            try:f.write(p,arcname=rel)
            except Exception as e:errors.append('Paquet '+rel+': '+str(e)[:100])
    if zip_path.stat().st_size>250*2**20:errors.append('Paquet trop volumineux pour le connecteur : '+zip_path.name)

# Résumé + contrôle couverture.
sc=Counter(row[0].split('/')[0] for row in records)
summary={
    'archive_source':archive.name,'archive_size_bytes':archive.stat().st_size,
    'images':image_stats,'nombre_fichiers_inventories':len(records),'nombre_avec_indices':len(hits_rows),
    'nombre_archives_apk_jar':len(app_rows),'nombre_entrees_dans_archives':len(archive_rows),
    'nombre_entrees_archive_avec_mentions':len(archive_hits),'fichiers_par_partition':dict(sc),
    'nombre_erreurs_ou_limites':len(errors),'avertissements':errors[:400],
    'important':'Analyse statique ; chaque fichier extrait est inventorié, pas de code système exécuté ; aucune preuve de compatibilité sans test sur IHU007B.'
}
safe_write('synthese.json',json.dumps(summary,ensure_ascii=False,indent=2))
safe_write('erreurs_et_limites.txt','\n'.join(errors) if errors else 'Aucune erreur signalée durant les étapes couvertes.')
package=reports/'RAPPORTS_IHU007B_COMPLET.zip'
with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(reports.iterdir()):
        if p!=package and p.is_file() and not p.name.startswith('composants_techniques_'):
            z.write(p,p.name)
folder=mydrive/'IHU007_ANALYSE_RAPPORTS'/report_id
folder.mkdir(parents=True,exist_ok=True)
for p in [package]+sorted(reports.glob('composants_techniques_*.zip')):
    if p.stat().st_size>250*2**20:
        print('Trop gros pour le connecteur ; fichier non transféré :',p.name);continue
    shutil.copy2(p,folder/p.name)
print('\n===== ANALYSE AUTOMATISÉE TERMINÉE =====')
print('Fichiers analysés :',len(records),' ; APK/JAR/ZIP :',len(app_rows),' ; entrées internes :',len(archive_rows))
print('Rapports enregistrés :',folder)
print('Erreurs/limites :',len(errors),' ; vérifier synthese.json pour la couverture réelle.')
print('Retourner dans ChatGPT et écrire : « analyse IHU007 terminée, lis les rapports dans Drive »')
